# 🧙‍♀️ Merlina on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Schneewolf-Labs/Merlina/blob/main/notebooks/Merlina_Colab.ipynb)

Run the full Merlina training UI on a free Colab GPU. Three cells: install, launch, open.

**Before you start:** `Runtime → Change runtime type → GPU` (a free T4 has 16 GB — enough for 4-bit QLoRA on models up to ~7B).

**Heads up:** Colab storage is ephemeral. Push finished models to the HuggingFace Hub from the Export tab (set your `HF_TOKEN` in the UI) before the session ends.

In [ ]:
# 📦 Install Merlina (~2-3 min). Colab ships a CUDA-matched torch already,
# which requirements.txt deliberately leaves untouched.
%cd /content
!git clone -q https://github.com/Schneewolf-Labs/Merlina.git 2>/dev/null || git -C Merlina pull -q
%cd /content/Merlina
!pip install -q -r requirements.txt

import torch
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU — switch runtime type to GPU before training!")

In [ ]:
# ✨ Launch the Merlina server in the background
import subprocess, time, urllib.request

log = open("/content/merlina.log", "w")
server = subprocess.Popen(
    ["python", "merlina.py"], cwd="/content/Merlina",
    stdout=log, stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("✅ Merlina is up!")
        break
    except Exception:
        if server.poll() is not None:
            print("❌ Server exited — last log lines:")
            print(open("/content/merlina.log").read()[-3000:])
            break
        time.sleep(2)
else:
    print("⏱️ Timed out — check /content/merlina.log")

In [ ]:
# 🔮 Open the UI (served through Colab's built-in port proxy)
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
print(f"🧙‍♀️ Click to open Merlina: {url}")

## Tips for Colab training

- **Enable 4-bit quantization** in the training form — a T4 can't fit most models in bf16.
- **Start small**: `Qwen/Qwen2.5-1.5B-Instruct` or similar 1–3B models train comfortably; 7B works in 4-bit with small batch sizes.
- **Push to the Hub**: set `push_to_hub` with your HF token so the finished model outlives the session.
- **Watch progress here** if you like: run `!tail -n 30 /content/merlina.log` in a new cell.
- Pre-flight validation runs before every job and will warn about VRAM problems before any GPU time is wasted.

In [ ]:
# 🛑 Optional: stop the server
# server.terminate()